In [ ]:
# run this notebook after 6_find_differences_twins_python
# use this notebook to merge twin difference files and identify error-prone giab difficult regions
# make sure to download giab difficult regions (v3.6) from 
# https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/genome-stratifications/v3.6/
# we also need ukb twin differences for the giab analysis (in this case, in /ukb/twins_diffs_full.txt)
# after this, run 8_analyze_twin_spectra_python

In [ ]:
library(dplyr)
library(data.table)
library(ggplot2)
library(bedtoolsr)
library(zoo) 

In [ ]:
twin_dup = fread("./relatedness_dec20/twin_dup.txt", sep = "\t")
flagged = fread("./flagged_samples.tsv", sep = "\t", header= TRUE)
twin_dup = twin_dup %>% filter(!(IID1 %in% flagged$s) & !(IID2 %in% flagged$s))

In [ ]:
format_difs_no_gq = function(difs){
    names(difs) = c("locus", "alleles", "idv1_gt", "idv2_gt")
    difs$chr = NA
    difs$pos = NA
    chr_pos = gsub(x = difs$locus, pattern = "chr", replace = "")
    difs$chr = gsub(x = chr_pos, pattern = ":.*", replace = "")
    difs$pos = gsub(x = chr_pos, pattern = ".*:", replace = "") 
    difs = difs %>% filter(chr %in% 1:22)
    return(difs)
}

In [ ]:
is_00_11 = function(i1_gt, i2_gt){
    return(i1_gt %in% c("1/1", "1|1") | i2_gt %in% c("1/1", "1|1"))
}

In [ ]:
# first, merge all twin/duplicate difs files 
twin_dup$file_exists = NA
for (i in 1:nrow(twin_dup)){
    if (i %% 100 == 0){
        message(i)
    }
    idv1 = twin_dup$IID1[i]
    idv2 = twin_dup$IID2[i]
    f = paste("./twins_dups/difs/", idv1, "_", idv2, "_difs.tsv", sep = "")
    f_all_alt = paste("./twins_dups/difs_all_alt/", idv1, "_", idv2, "_difs_all_alt.tsv", sep = "")
    f_out = paste("./twins_dups/formatted_difs/", idv1, "_", idv2, "_difs_formatted.tsv", sep = "")
    if (file.exists(f) & file.exists(f_all_alt)){
        twin_dup$file_exists[i] = TRUE
        d = fread(f, sep = "\t", header = TRUE) # all 0/0 0/1 and 0/0 1/1 columns 
        formatted = format_difs_no_gq(d)  
        d_all_alt = fread(f_all_alt, sep = "\t", header = TRUE)
        # remove any locus seen more than once
        duplicated_indices = which(duplicated(formatted$locus) | duplicated(formatted$locus, fromLast = TRUE))
        formatted = formatted[-duplicated_indices,]
        # remove any locus not in the format 0/0 0/1 
        formatted = formatted[which(!mapply(is_00_11, formatted$idv1_gt, formatted$idv2_gt)),]
        # remove any locus seen more than once in the difs_all_alt file 
        formatted$remove = FALSE
        cnt = table(d_all_alt$locus)
        formatted$remove = (cnt[match(formatted$locus, names(cnt))] > 1)
        formatted$remove[is.na(formatted$remove)] = FALSE
        formatted = formatted[!formatted$remove,]
        fwrite(formatted, f_out, sep = "\t", quote = FALSE, row.names = FALSE)  
    } else {
        twin_dup$file_exists[i] = FALSE
    }
}

In [ ]:
twin_dup$file_exists = NA
twin_dup$num_snps = NA
twin_dup$num_indels = NA

for (i in 1:1){
    idv1 = twin_dup$IID1[i]
    idv2 = twin_dup$IID2[i]
    f = paste("./twins_dups/formatted_difs/", idv1, "_", idv2, "_difs_formatted.tsv", sep = "")
    if (file.exists(f)){
        difs = fread(f, sep = "\t", header = TRUE)
        snps = difs %>% filter(nchar(alleles) == 9) %>% select(locus, alleles, chr, pos, idv1_gt, idv2_gt)
        twin_dup$file_exists[i] = TRUE
        twin_dup$num_snps[i] = nrow(snps)
        twin_dup$num_indels[i] = nrow(difs) - nrow(snps)
    } else {
        twin_dup$file_exists[i] = FALSE
    }
}


all_snps = snps 
all_snps$idv1 = idv1
all_snps$idv2 = idv2

for (i in 2:nrow(twin_dup)){
    if (i %% 100 == 0){
        message(i)
    }
    idv1 = twin_dup$IID1[i]
    idv2 = twin_dup$IID2[i]
    f = paste("./twins_dups/formatted_difs/", idv1, "_", idv2, "_difs_formatted.tsv", sep = "")
    if (file.exists(f)){
        difs = fread(f, sep = "\t", header = TRUE)
        snps = difs %>% filter(nchar(alleles) == 9) %>% select(locus, alleles, chr, pos, idv1_gt, idv2_gt)
        twin_dup$file_exists[i] = TRUE
        twin_dup$num_snps[i] = nrow(snps)
        twin_dup$num_indels[i] = nrow(difs) - nrow(snps)
        snps$idv1 = idv1 
        snps$idv2 = idv2
        all_snps = rbind(all_snps, snps)
    } else {
        twin_dup$file_exists[i] = FALSE
    }
}

fwrite(all_snps, "./twins_dups_all_snps_after_standard_QC_oct16.txt", sep = "\t",
      row.names = FALSE, col.names = TRUE, quote = FALSE)

In [ ]:
# read in aou and ukb 
aou_snps = fread("./twins_dups_all_snps_after_standard_QC_oct16.txt", sep = "\t")
ukb_difs = fread("./ukb/twins_diffs_full.txt", sep=" ") 
# add twin info to ukbb
ukb_difs$idv = NA
id = 0
prev = 100
for (i in 1:nrow(ukb_difs)){
    if (ukb_difs$CHROM[i] < prev){
        id = id + 1
    }
    ukb_difs$idv[i] = id
    prev = ukb_difs$CHROM[i]
}
ukb_snps = ukb_difs %>% filter(nchar(REF) == 1 & nchar(ALT) == 1)
ukb_snps$chr = ukb_snps$CHROM
ukb_snps$pos = ukb_snps$POS

In [ ]:
chr_lens = fread("./hg38.chrom.sizes", sep = "\t")
chr_lens$chr = gsub(chr_lens$V1, pattern = "chr", replace = "")
chr_lens = chr_lens %>% filter(chr %in% 1:22)
chr_lens$start = 1 
chr_lens$end = chr_lens$V2
chr_lens$length = chr_lens$V2
genome = sum(chr_lens$length)

In [ ]:
# all in giab difficult regions 

giab_difficult = c("./giab_3.6/GRCh38_lowmappabilityall.bed.gz",
"./giab_3.6/GRCh38_segdups.bed.gz",
# tandem repeats included in giab difficult regions
#"./giab_3.6/GRCh38_AllTandemRepeatsandHomopolymers_slop5.bed.gz",
"./giab_3.6/GRCh38_AllTandemRepeats_le50bp_slop5.bed.gz",
"./giab_3.6/GRCh38_AllTandemRepeats_51to200bp_slop5.bed.gz",
"./giab_3.6/GRCh38_AllTandemRepeats_201to10000bp_slop5.bed.gz",
"./giab_3.6/GRCh38_AllTandemRepeats_ge10001bp_slop5.bed.gz",
"./giab_3.6/GRCh38_satellites_slop5.bed.gz",
"./giab_3.6/GRCh38_AllHomopolymers_ge7bp_imperfectge11bp_slop5.bed.gz",
# 25 included in giab difficult
"./giab_3.6/GRCh38_gclt25orgt65_slop50.bed.gz",
# functionally technically difficult 
"./giab_3.6/GRCh38_BadPromoters.bed.gz",                                   
"./giab_3.6/GRCh38_CMRGv1.00_duplicationinKMT2C.bed.gz",
"./giab_3.6/GRCh38_CMRGv1.00_falselyduplicatedgenes.bed.gz",    
# all other difficult 
"./giab_3.6/GRCh38_KIR.bed.gz",  
"./giab_3.6/GRCh38_collapsed_duplication_FP_regions.bed.gz", 
"./giab_3.6/GRCh38_population_CNV_FP_regions.bed.gz",  
"./giab_3.6/GRCh38_false_duplications_correct_copy.bed.gz", 
"./giab_3.6/GRCh38_false_duplications_incorrect_copy.bed.gz",
"./giab_3.6/GRCh38_LD_discordant_haplotypes_slop5bp.bed.gz",  
"./giab_3.6/GRCh38_gnomAD_InbreedingCoeff_slop1bp_merge1000bp.bed.gz",   
"./giab_3.6/GRCh38_L1H_gt500.bed.gz",
"./giab_3.6/GRCh38_MHC.bed.gz",  
"./giab_3.6/GRCh38_VDJ.bed.gz",  
"./giab_3.6/GRCh38_contigs_lt500kb.bed.gz", 
"./giab_3.6/GRCh38_gaps_slop15kb.bed.gz") 

In [ ]:
aou_bed = data.frame(chrom = aou_snps$chr, start = aou_snps$pos, end = aou_snps$pos+1)
ukb_bed = data.frame(chrom = ukb_snps$chr, start = ukb_snps$pos, end = ukb_snps$pos+1)

In [ ]:
# make a file with what percent of the genome each stratification represents 
stratification_genome = data.frame(stratification = giab_difficult, coverage = NA, aou_twin_duplicate_snps = NA,
                                  ukb_twin_duplicate_snps=NA, aou_p = NA, ukb_p = NA, aou_ratio = NA,
                                  ukb_ratio = NA)
for (i in 1:nrow(stratification_genome)){
    message(stratification_genome$stratification[i])
    strat = fread(stratification_genome$stratification[i])
    names(strat) = c("chr_prev", "start", "end")
    strat$chrom = gsub(x = strat$chr_prev, pattern = "chr", replace = "")
    strat$length = strat$end - strat$start + 1
    strat = strat %>% filter(chrom %in% 1:22)
    stratification_genome$coverage[i] = sum(strat$length)/genome
    giab_bed = strat[,c("chrom", "start", "end")]
    aou_giab = bedtoolsr::bt.intersect(a = aou_bed, b = giab_bed, c = TRUE)
    ukbb_giab = bedtoolsr::bt.intersect(a = ukb_bed, b = giab_bed, c = TRUE)
    names(aou_giab) = c("chr", "pos", "pos2", "in_giab")
    names(ukbb_giab) = c("chr", "pos", "pos2", "in_giab")
    stratification_genome$aou_twin_duplicate_snps[i] = sum(aou_giab$in_giab == 1)
    stratification_genome$ukb_twin_duplicate_snps[i] = sum(ukbb_giab$in_giab == 1)
    
    stratification_genome$aou_p[i] = binom.test(x = stratification_genome$aou_twin_duplicate_snps[i], 
                                               n = nrow(aou_bed), p = stratification_genome$coverage[i],
                                               alternative = "greater")$p.value
    stratification_genome$ukb_p[i] = binom.test(x = stratification_genome$ukb_twin_duplicate_snps[i], 
                                               n = nrow(ukb_bed), p = stratification_genome$coverage[i],
                                               alternative = "greater")$p.value
    stratification_genome$aou_ratio[i] = stratification_genome$aou_twin_duplicate_snps[i]/nrow(aou_bed)/stratification_genome$coverage[i]
    stratification_genome$ukb_ratio[i] = stratification_genome$ukb_twin_duplicate_snps[i]/nrow(ukb_bed)/stratification_genome$coverage[i]

}

In [ ]:
adjusted = p.adjust(c(stratification_genome$aou_p, stratification_genome$ukb_p), method = "fdr")
stratification_genome$aou_fdr = adjusted[1:nrow(stratification_genome)]
stratification_genome$ukb_fdr=adjusted[(nrow(stratification_genome)+1):length(adjusted)]

In [ ]:
fwrite(stratification_genome, "./giab_strat_stats_oct27.tsv", sep = "\t", col.names=TRUE, row.names=FALSE, quote=FALSE)

In [ ]:
to_include = stratification_genome$stratification[which(stratification_genome$aou_fdr <= 0.05 | 
                                                       stratification_genome$ukb_fdr <= 0.05 )]

In [ ]:
all_union = fread(to_include[1], header=FALSE)
names(all_union) = c("chrom", "start", "end")
all_union = all_union %>% filter(chrom %in% paste("chr", 1:22, sep = ""))
all_union$chrom = as.numeric(gsub(x = all_union$chrom, pattern = "chr", replace = ""))
all_union$start = as.numeric(all_union$start)
all_union$end = as.numeric(all_union$end)
all_union = all_union %>% filter(chrom %in% 1:22)
for (i in 2:length(to_include)){
    message(i)
    add = fread(to_include[i], header=FALSE)
    names(add) = c("chrom", "start", "end")
    add = add %>% filter(chrom %in% paste("chr", 1:22, sep = ""))
    add$chrom = as.numeric(gsub(x =  add$chrom, pattern = "chr", replace = ""))
    add$start = as.numeric(add$start)
    add$end = as.numeric(add$end)
    sorted = rbind(all_union, add)
    sorted = sorted %>% arrange(chrom, start)
    all_union = bedtoolsr::bt.merge(i = sorted, d = 0)
    names(all_union) = c("chrom", "start", "end")
}

In [ ]:
fwrite(all_union, "./giab_difficult_merged_oct27.bed", sep = "\t", quote = FALSE, row.names=FALSE,col.names=TRUE)

In [ ]:
ukb_snps = ukb_snps %>% filter(in_giab == 0)
aou_snps = aou_snps %>% filter(in_giab == 0)
fwrite(ukb_snps, "ukb_twin_differences_giab_removed.txt", sep = "\t", 
      row.names = FALSE, quote = FALSE)
fwrite(aou_snps, "aou_twin_differences_giab_removed.txt", sep = "\t", 
      row.names = FALSE, quote = FALSE)